# U.S. Airline Operational Performance Analysis

**Author**: Ashwin Parthasarathy

**Dataset**: U.S. Bureau of Transportation Statistics - Reporting Carrier On-Time Performance

This project builds a reproducible Python data workflow to analyze recent U.S. airline operational performance. The analysis focuses on flight delays, cancellations, airports, carriers, routes, and temporal patterns using recent BTS flight operations data.

## 1. Setup
---

In [7]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [8]:
pd.set_option("display.max_columns", None)
print("Numpy: ", np.__version__)
print("Pandas: ", pd.__version__)

Numpy:  2.5.3
Pandas:  3.0.6


## 2. Data Ingestion
---

### Dataset Source and Scope

This analysis uses the U.S. Bureau of Transportation Statistics (BTS)
Reporting Carrier On-Time Performance dataset.

The working dataset covers January 2025 through July 2026. Raw monthly
BTS ZIP archives are downloaded reproducibly and converted to focused
Parquet files containing the operational fields required for this analysis.

The raw and processed datasets are excluded from Git because of their size.
The repository instead includes scripts to reproduce the local dataset.

In [9]:
PROJECT_ROOT = Path.cwd()
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"

parquet_files = sorted(PROCESSED_DATA_DIR.glob("bts_flights_*.parquet"))

print(f"Monthly files found: {len(parquet_files)}")
print(f"First file: {parquet_files[0].name}")
print(f"Last file:  {parquet_files[-1].name}")

Monthly files found: 19
First file: bts_flights_2025_01.parquet
Last file:  bts_flights_2026_07.parquet


In [11]:
# Take inventory of all files and inspect for data sufficiency (no. of records, start date, end date)
monthly_inventory = []

for path in parquet_files:
    dates = pd.read_parquet(
        path,
        columns=["FlightDate"],
    )

    monthly_inventory.append(
        {
            "file": path.name,
            "rows": len(dates),
            "start_date": dates["FlightDate"].min(),
            "end_date": dates["FlightDate"].max(),
        }
    )

inventory_df = pd.DataFrame(monthly_inventory)
inventory_df

,file,rows,start_date,end_date
0,bts_flights_2025_01.parquet,539747,2025-01-01,2025-01-31
1,bts_flights_2025_02.parquet,504884,2025-02-01,2025-02-28
2,bts_flights_2025_03.parquet,600872,2025-03-01,2025-03-31
3,bts_flights_2025_04.parquet,583950,2025-04-01,2025-04-30
4,bts_flights_2025_05.parquet,605648,2025-05-01,2025-05-31
5,bts_flights_2025_06.parquet,611575,2025-06-01,2025-06-30
6,bts_flights_2025_07.parquet,631428,2025-07-01,2025-07-31
7,bts_flights_2025_08.parquet,602378,2025-08-01,2025-08-31
8,bts_flights_2025_09.parquet,562439,2025-09-01,2025-09-30
9,bts_flights_2025_10.parquet,605844,2025-10-01,2025-10-31


In [12]:
print(f"Total rows: {inventory_df['rows'].sum():,}")
print("Date range: ", inventory_df['start_date'].min(), " to ", inventory_df['end_date'].max())

Total rows: 11,121,962
Date range:  2025-01-01 00:00:00  to  2026-07-31 00:00:00


In [13]:
# Sample one month's data
sample_df = pd.read_parquet(parquet_files[0])

print(f"Shape: {sample_df.shape}")
sample_df.head()

Shape: (539747, 41)


,Year,Month,DayOfWeek,FlightDate,Reporting_Airline,DOT_ID_Reporting_Airline,Flight_Number_Reporting_Airline,OriginAirportID,Origin,OriginCityName,OriginState,DestAirportID,Dest,DestCityName,DestState,CRSDepTime,DepTime,DepDelay,DepDelayMinutes,DepDel15,DepTimeBlk,TaxiOut,TaxiIn,CRSArrTime,ArrTime,ArrDelay,ArrDelayMinutes,ArrDel15,ArrTimeBlk,Cancelled,CancellationCode,Diverted,CRSElapsedTime,ActualElapsedTime,AirTime,Distance,CarrierDelay,WeatherDelay,NASDelay,SecurityDelay,LateAircraftDelay
0,2025,1,3,2025-01-01,AA,19805,1,12478,JFK,"New York, NY",NY,12892,LAX,"Los Angeles, CA",CA,659,656.0,-3.0,0.0,0.0,0600-0659,23.0,9.0,1020,1013.0,-7.0,0.0,0.0,1000-1059,0.0,NaN,0.0,381.0,377.0,345.0,2475.0,NaN,NaN,NaN,NaN,NaN
1,2025,1,4,2025-01-02,AA,19805,1,12478,JFK,"New York, NY",NY,12892,LAX,"Los Angeles, CA",CA,659,652.0,-7.0,0.0,0.0,0600-0659,30.0,7.0,1020,1022.0,2.0,2.0,0.0,1000-1059,0.0,NaN,0.0,381.0,390.0,353.0,2475.0,NaN,NaN,NaN,NaN,NaN
2,2025,1,5,2025-01-03,AA,19805,1,12478,JFK,"New York, NY",NY,12892,LAX,"Los Angeles, CA",CA,659,652.0,-7.0,0.0,0.0,0600-0659,13.0,11.0,1020,1003.0,-17.0,0.0,0.0,1000-1059,0.0,NaN,0.0,381.0,371.0,347.0,2475.0,NaN,NaN,NaN,NaN,NaN
3,2025,1,6,2025-01-04,AA,19805,1,12478,JFK,"New York, NY",NY,12892,LAX,"Los Angeles, CA",CA,700,653.0,-7.0,0.0,0.0,0700-0759,24.0,10.0,1026,1016.0,-10.0,0.0,0.0,1000-1059,0.0,NaN,0.0,386.0,383.0,349.0,2475.0,NaN,NaN,NaN,NaN,NaN
4,2025,1,7,2025-01-05,AA,19805,1,12478,JFK,"New York, NY",NY,12892,LAX,"Los Angeles, CA",CA,659,655.0,-4.0,0.0,0.0,0600-0659,26.0,5.0,1020,1013.0,-7.0,0.0,0.0,1000-1059,0.0,NaN,0.0,381.0,378.0,347.0,2475.0,NaN,NaN,NaN,NaN,NaN


In [14]:
sample_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 539747 entries, 0 to 539746
Data columns (total 41 columns):
 #   Column                           Non-Null Count   Dtype         
---  ------                           --------------   -----         
 0   Year                             539747 non-null  int64         
 1   Month                            539747 non-null  int64         
 2   DayOfWeek                        539747 non-null  int64         
 3   FlightDate                       539747 non-null  datetime64[us]
 4   Reporting_Airline                539747 non-null  str           
 5   DOT_ID_Reporting_Airline         539747 non-null  int64         
 6   Flight_Number_Reporting_Airline  539747 non-null  int64         
 7   OriginAirportID                  539747 non-null  int64         
 8   Origin                           539747 non-null  str           
 9   OriginCityName                   539747 non-null  str           
 10  OriginState                      539747 non-null  str  

In [18]:
# Summary of dataset

print(f"Monthly files: {len(parquet_files)}")
print(f"Total rows: {inventory_df['rows'].sum()}")
print(f"Date range: {inventory_df['start_date'].min()} to {inventory_df['end_date'].max()}")

Monthly files: 19
Total rows: 11121962
Date range: 2025-01-01 00:00:00 to 2026-07-31 00:00:00


> The working dataset contains 11.12M flight records across 19 monthly partitions. To keep the workflow reproducible and memory-efficient, the analysis processes monthly parquet partitions rather than loading the entire dataset into memory at once.

## 3. Data Cleaning
---
Before applying any cleaning rules, the dataset is assessed for missing values, unexpected values, and fields whose missingness is structurally expected.

This distinction is important for flight operations data because cancelled or diverted flights may legitimately lack arrival or elapsed-time measurements. Cleaning rules are therefore based on the operational meaning of each field rather than treating every missing value as an error.

### Data Quality Assessment

In [19]:
missing_summary = (
    sample_df.isna()
    .sum()
    .sort_values(ascending=False)
    .to_frame("missing_count")
)

missing_summary["missing_pct"] = missing_summary["missing_count"]/len(sample_df) * 100

missing_summary

,missing_count,missing_pct
CancellationCode,523435,96.977843
WeatherDelay,441617,81.819260
CarrierDelay,441617,81.819260
LateAircraftDelay,441617,81.819260
NASDelay,441617,81.819260
SecurityDelay,441617,81.819260
ArrDelay,17478,3.238184
AirTime,17478,3.238184
ActualElapsedTime,17478,3.238184
ArrDelayMinutes,17478,3.238184


> - `CancellationCode` is missing for 96.98% of rows. This is likely correct because this field should be populated only for meaningful flight cancellations.
> - All five delay cause fields have exactly the same missing counts (441,617 - 81.82%). This indicates that these fields are populated under specific operational conditions.
> - Departure fields such as `DepTime`, `DepDelay`, `DepDel15` are missing for about 2.95% of flights.
> - Arrival-related fields are missing slightly more often, around 3.24%.
> - Core schedule and identify fields (airline, airport, date, distance, scheduled time) do not have missing values. This is excellent for the analysis.

In [24]:
columns_to_check = [
"Cancelled",
"CancellationCode",
"Diverted",
"DepDelay",
"DepDelayMinutes",
"DepDel15",
"ArrDelay",
"ArrDelayMinutes",
"ArrDel15",
"CarrierDelay",
"WeatherDelay",
"NASDelay",
"SecurityDelay",
"LateAircraftDelay"
]

for column in columns_to_check:
    print("\n")
    print(sample_df[column].value_counts(dropna=False).head(15))



Cancelled
0.0    523435
1.0     16312
Name: count, dtype: int64


CancellationCode
NaN    523435
B       14327
A        1635
C         342
D           8
Name: count, dtype: int64


Diverted
0.0    538581
1.0      1166
Name: count, dtype: int64


DepDelay
-5.0     38577
-6.0     34950
-4.0     34842
-3.0     32288
-7.0     30571
-2.0     28830
-8.0     25320
-1.0     25308
-10.0    22097
 0.0     21405
-9.0     21198
 NaN     15923
 1.0     10725
-11.0     9974
 2.0      8898
Name: count, dtype: int64


DepDelayMinutes
0.0     346716
NaN      15923
1.0      10725
2.0       8898
3.0       7993
4.0       7206
5.0       6574
6.0       5916
7.0       5410
8.0       4970
9.0       4704
10.0      4398
11.0      3860
12.0      3803
13.0      3512
Name: count, dtype: int64


DepDel15
0.0    428164
1.0     95660
NaN     15923
Name: count, dtype: int64


ArrDelay
 NaN     17478
-13.0    13972
-15.0    13868
-14.0    13810
-12.0    13698
-11.0    13425
-16.0    13262
-10.0    13229
-9.0     1301

In [25]:
# Examine whether missing operational values are associated with cancellations and diversions

operational_missingness = pd.DataFrame(
    {
        "total_rows": [len(sample_df), len(sample_df), len(sample_df)],
        "condition_rows": [
            (sample_df["Cancelled"] == 1).sum(),
            (sample_df["Diverted"] == 1).sum(),
            ((sample_df["Cancelled"] == 0) & (sample_df["Diverted"]==0)).sum()
        ]
    },
    index = ["Cancelled flights", "Diverted flights", "Completed non-diverted flights"]
)

operational_missingness

,total_rows,condition_rows
Cancelled flights,539747,16312
Diverted flights,539747,1166
Completed non-diverted flights,539747,522269


In [29]:
fields_to_check = [
    "DepTime",
    "DepDelay",
    "ArrTime",
    "ArrDelay",
    "ActualElapsedTime",
    "AirTime",
    "CancellationCode",
]

for column in fields_to_check:
    print(f"\n ***** {column} *****")

    print(
        pd.crosstab(
            [
                sample_df["Cancelled"],
                sample_df["Diverted"],
            ],
            sample_df[column].isna(),
            rownames=["Cancelled", "Diverted"],
            colnames=["Missing"],
        )
    )


 ***** DepTime *****
Missing              False  True 
Cancelled Diverted               
0.0       0.0       522269      0
          1.0         1166      0
1.0       0.0          426  15886

 ***** DepDelay *****
Missing              False  True 
Cancelled Diverted               
0.0       0.0       522269      0
          1.0         1166      0
1.0       0.0          389  15923

 ***** ArrTime *****
Missing              False  True 
Cancelled Diverted               
0.0       0.0       522269      0
          1.0          898    268
1.0       0.0            0  16312

 ***** ArrDelay *****
Missing              False  True 
Cancelled Diverted               
0.0       0.0       522269      0
          1.0            0   1166
1.0       0.0            0  16312

 ***** ActualElapsedTime *****
Missing              False  True 
Cancelled Diverted               
0.0       0.0       522269      0
          1.0            0   1166
1.0       0.0            0  16312

 ***** AirTime *****
Missin

In [31]:
pd.crosstab(
    sample_df["Cancelled"],
    sample_df["CancellationCode"].isna(),
    rownames=["Cancelled"],
    colnames=["CancellationCode missing"],
)

CancellationCode missing,False,True
Cancelled,,
0.0,0,523435
1.0,16312,0


In [30]:
delay_cause_columns = [
    "CarrierDelay",
    "WeatherDelay",
    "NASDelay",
    "SecurityDelay",
    "LateAircraftDelay",
]

for column in delay_cause_columns:
    print(f"\n***** {column} *****")
    print(
        pd.crosstab(
            sample_df["ArrDel15"],
            sample_df[column].isna(),
            rownames=["ArrDel15"],
            colnames=["Missing"],
            dropna=False,
        )
    )


***** CarrierDelay *****
Missing   False   True 
ArrDel15               
0.0           0  424139
1.0       98130       0
NaN           0   17478

***** WeatherDelay *****
Missing   False   True 
ArrDel15               
0.0           0  424139
1.0       98130       0
NaN           0   17478

***** NASDelay *****
Missing   False   True 
ArrDel15               
0.0           0  424139
1.0       98130       0
NaN           0   17478

***** SecurityDelay *****
Missing   False   True 
ArrDel15               
0.0           0  424139
1.0       98130       0
NaN           0   17478

***** LateAircraftDelay *****
Missing   False   True 
ArrDel15               
0.0           0  424139
1.0       98130       0
NaN           0   17478


**Missing-Value Interpretation**

> The missing-value analysis indicates that most missing operational values are
structural rather than random data-quality problems.

> Cancellation codes are populated only for cancelled flights. Likewise,
arrival-delay and elapsed-time fields are unavailable for cancelled and
diverted flights because those flights did not complete the originally
scheduled itinerary normally. Delay-cause fields are populated only for
flights classified as arriving at least 15 minutes late.

> These missing values are therefore preserved rather than automatically
imputed or removed. Subsequent analyses filter flights according to the
operational question being evaluated.

In [32]:
quality_checks = {
    "duplicate_rows": sample_df.duplicated().sum(),
    "missing_flight_date": sample_df["FlightDate"].isna().sum(),
    "invalid_month": (~sample_df["Month"].between(1, 12)).sum(),
    "invalid_day_of_week": (~sample_df["DayOfWeek"].between(1, 7)).sum(),
    "nonpositive_distance": (sample_df["Distance"] <= 0).sum(),
    "negative_taxi_out": (sample_df["TaxiOut"] < 0).sum(),
    "negative_taxi_in": (sample_df["TaxiIn"] < 0).sum(),
    "nonpositive_scheduled_elapsed": (
        sample_df["CRSElapsedTime"] <= 0
    ).sum(),
    "invalid_cancelled_flag": (
        ~sample_df["Cancelled"].isin([0, 1])
    ).sum(),
    "invalid_diverted_flag": (
        ~sample_df["Diverted"].isin([0, 1])
    ).sum(),
}

pd.Series(quality_checks, name="count")

duplicate_rows                   0
missing_flight_date              0
invalid_month                    0
invalid_day_of_week              0
nonpositive_distance             0
negative_taxi_out                0
negative_taxi_in                 0
nonpositive_scheduled_elapsed    0
invalid_cancelled_flag           0
invalid_diverted_flag            0
Name: count, dtype: int64

In [33]:
date_consistency = pd.Series(
    {
        "year_mismatch": (
            sample_df["FlightDate"].dt.year
            != sample_df["Year"]
        ).sum(),
        "month_mismatch": (
            sample_df["FlightDate"].dt.month
            != sample_df["Month"]
        ).sum(),
    },
    name="count",
)

date_consistency

year_mismatch     0
month_mismatch    0
Name: count, dtype: int64

In [34]:
sample_df[
    ["CRSDepTime", "CRSArrTime"]
].agg(["min", "max"])

,CRSDepTime,CRSArrTime
min,5,1
max,2359,2359


### Cleaning Strategy

The initial quality asssessment found no exact duplicates, invalid core identifiers, unreasonable distance values, or inconsistencies between flight date, year/month fields in the first month (Jan 2025) partition.

Most missing values were found to be structurally meaningful and were therefore preserved. Cleaning is intentionally conservative.
- Exact duplicate records are removed if encountered
- Key fields are standardized to appropriate data types

These rules can be applied consistently to every monthly partition in the dataset.

In [35]:
def remove_duplicate_records(df):
    """Remove exact duplicate flight records.
    
    Parameters
    ----------
    df: pandas.DataFrame
        Flight records to clean.
        
    Returns
    -------
    pandas.DataFrame
        Copy of the input data with exact duplicate rows removed.
    """
    return df.drop_duplicates().copy()

In [40]:
def standardize_data_types(df):
    """Standardize data types used in the flight dataset.

    Parameters
    ----------
    df: pandas.DataFrame
        Flight records to clean.

    Returns
    -------
    pandas.DataFrame
        Copy of the input data with binary operational fields converted to nullable boolean types.
    """
    cleaned = df.copy()

    cleaned["Cancelled"] = cleaned["Cancelled"].astype("boolean")
    cleaned["Diverted"] = cleaned["Diverted"].astype("boolean")
    cleaned["DepDel15"] = cleaned["DepDel15"].astype("boolean")
    cleaned["ArrDel15"] = cleaned["ArrDel15"].astype("boolean")

    return cleaned

In [41]:
cleaned_sample_df = remove_duplicate_records(sample_df)
cleaned_sample_df = standardize_data_types(cleaned_sample_df)

print(f"Rows before cleaning: {len(sample_df)}")
print(f"Rows after cleaning: {len(cleaned_sample_df)}")

cleaned_sample_df[["Cancelled", "Diverted", "DepDel15", "ArrDel15"]].dtypes

Rows before cleaning: 539747
Rows after cleaning: 539747


Cancelled    boolean
Diverted     boolean
DepDel15     boolean
ArrDel15     boolean
dtype: object

In [39]:
cleaned_sample_df[
    [
        "CancellationCode",
        "ArrDelay",
        "CarrierDelay",
        "WeatherDelay",
        "NASDelay",
        "SecurityDelay",
        "LateAircraftDelay",
    ]
].isna().sum()

CancellationCode     523435
ArrDelay              17478
CarrierDelay         441617
WeatherDelay         441617
NASDelay             441617
SecurityDelay        441617
LateAircraftDelay    441617
dtype: int64

## 4. Exploratory Data Analysis
---

## 5. Visualizations
---

## 6. Summary & Interpretation
---